**Goal:** Find Gonality Savings Graphs (V=7, max edge multiplicity 2) using an autoregressive transformer trained on random valid graphs, then iteratively refined via a reward-guided generate → score → retrain loop. The goal of this script is to time this method on 10 trials.

Encoding: lower-triangle, row by row. First token = vertex count + 10. EOS = 4, PAD = 5.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import networkx as nx

NUM_VERTICES = 7
NUM_EDGES_UPPER = NUM_VERTICES * (NUM_VERTICES - 1) // 2  # 21
TARGET_GRAPHS = 3000
VERTEX_OFFSET = 10
EOS_TOKEN = 4
PAD_TOKEN = 5
EDGE_PROBS = [0.80, 0.15, 0.05]
SEQ_LENGTH = 1 + NUM_EDGES_UPPER + 1  # 23 tokens


def generate_random_graph(num_vertices=7, edge_probs=[0.80, 0.15, 0.05]):
    upper_tri_values = np.random.choice(
        [0, 1, 2], size=num_vertices * (num_vertices - 1) // 2, p=edge_probs
    )
    matrix = np.zeros((num_vertices, num_vertices), dtype=int)
    row_indices, col_indices = np.triu_indices(num_vertices, k=1)
    matrix[row_indices, col_indices] = upper_tri_values
    matrix[col_indices, row_indices] = upper_tri_values
    return matrix


def is_connected(matrix):
    binary = (matrix > 0).astype(int)
    G = nx.from_numpy_array(binary)
    return nx.is_connected(G)


def matrix_to_sequence(matrix, num_vertices=7):
    seq = [num_vertices + VERTEX_OFFSET]
    for i in range(1, num_vertices):
        row_connections = matrix[i, :i]
        seq.extend(row_connections.tolist())
    seq.append(EOS_TOKEN)
    return seq


def matrix_to_upper_triangle(matrix):
    row_indices, col_indices = np.triu_indices(matrix.shape[0], k=1)
    return matrix[row_indices, col_indices].tolist()


# Generate 3000 valid random graphs
print(f"Generating {TARGET_GRAPHS} valid random V={NUM_VERTICES} graphs...")
print(f"Edge probs: no_edge={EDGE_PROBS[0]}, single={EDGE_PROBS[1]}, double={EDGE_PROBS[2]}")
sequences = []
attempts = 0

while len(sequences) < TARGET_GRAPHS:
    matrix = generate_random_graph(NUM_VERTICES, EDGE_PROBS)
    attempts += 1
    if is_connected(matrix):
        seq = matrix_to_sequence(matrix, NUM_VERTICES)
        sequences.append(torch.tensor(seq, dtype=torch.long))
    if attempts % 5000 == 0:
        print(f"  ...{len(sequences)}/{TARGET_GRAPHS} valid after {attempts} attempts")

print(f"Done! {len(sequences)} valid graphs in {attempts} attempts ({len(sequences)/attempts*100:.1f}% acceptance)")

padded = [F.pad(seq, (0, SEQ_LENGTH - len(seq)), value=PAD_TOKEN) for seq in sequences]
dataset_tensor = torch.stack(padded)
print(f"Dataset shape: {dataset_tensor.shape}")

Generating 3000 valid random V=7 graphs...
Edge probs: no_edge=0.8, single=0.15, double=0.05
  ...535/3000 valid after 5000 attempts
  ...1114/3000 valid after 10000 attempts
  ...1683/3000 valid after 15000 attempts
  ...2230/3000 valid after 20000 attempts
  ...2778/3000 valid after 25000 attempts
Done! 3000 valid graphs in 26867 attempts (11.2% acceptance)
Dataset shape: torch.Size([3000, 23])


In [ ]:
import torch
import torch.nn as nn
import math

class GraphGPT(nn.Module):
    def __init__(self, vocab_size=6, max_seq_len=200, d_model=128, nhead=4, num_layers=4, pad_token=5):
        super(GraphGPT, self).__init__()
        self.pad_token = pad_token
        self.d_model = d_model
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_token)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_layer = nn.Linear(d_model, vocab_size)

    def generate_causal_mask(self, sz):
        return torch.triu(torch.ones(sz, sz), diagonal=1).bool()

    def forward(self, x):
        batch_size, seq_len = x.size()
        positions = torch.arange(0, seq_len, device=x.device).unsqueeze(0).expand(batch_size, seq_len)
        x_emb = self.token_embedding(x) + self.position_embedding(positions)
        x_emb = x_emb * math.sqrt(self.d_model)
        causal_mask = self.generate_causal_mask(seq_len).to(x.device)
        padding_mask = (x == self.pad_token)
        transformer_out = self.transformer(x_emb, mask=causal_mask, src_key_padding_mask=padding_mask)
        return self.output_layer(transformer_out)

In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast

vocab_size = 30
max_length = 23
pad_token = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_size = int(0.9 * len(dataset_tensor))
train_data, val_data = torch.utils.data.random_split(dataset_tensor, [train_size, len(dataset_tensor) - train_size])
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

model = GraphGPT(vocab_size=vocab_size, max_seq_len=max_length, pad_token=pad_token).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2, factor=0.5)
criterion = nn.CrossEntropyLoss(ignore_index=pad_token)
scaler = GradScaler()

epochs = 100
patience = 5
best_val_loss = float('inf')
epochs_no_improve = 0

for epoch in range(epochs):
    model.train()
    total_train_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        x, y = batch[:, :-1], batch[:, 1:]
        optimizer.zero_grad()
        with autocast():
            logits = model(x)
            loss = criterion(logits.reshape(-1, vocab_size), y.reshape(-1))
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_train_loss += loss.item()
    avg_train_loss = total_train_loss / len(train_loader)

    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            x_v, y_v = batch[:, :-1], batch[:, 1:]
            logits_v = model(x_v)
            loss_v = criterion(logits_v.reshape(-1, vocab_size), y_v.reshape(-1))
            total_val_loss += loss_v.item()
    avg_val_loss = total_val_loss / len(val_loader)
    scheduler.step(avg_val_loss)

    print(f"Epoch {epoch+1:03d} | Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'best_graph_gpt.pth')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print("Early stopping triggered.")
            break

/tmp/ipykernel_628/4242981412.py:20: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_628/4242981412.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 001 | Train: 1.2381 | Val: 0.8250 | LR: 0.000100
Epoch 002 | Train: 0.8247 | Val: 0.7961 | LR: 0.000100
Epoch 003 | Train: 0.8067 | Val: 0.7895 | LR: 0.000100
Epoch 004 | Train: 0.7983 | Val: 0.7834 | LR: 0.000100
Epoch 005 | Train: 0.7944 | Val: 0.7778 | LR: 0.000100
Epoch 006 | Train: 0.7897 | Val: 0.7740 | LR: 0.000100
Epoch 007 | Train: 0.7879 | Val: 0.7714 | LR: 0.000100
Epoch 008 | Train: 0.7850 | Val: 0.7721 | LR: 0.000100
Epoch 009 | Train: 0.7831 | Val: 0.7687 | LR: 0.000100
Epoch 010 | Train: 0.7826 | Val: 0.7698 | LR: 0.000100
Epoch 011 | Train: 0.7804 | Val: 0.7704 | LR: 0.000100
Epoch 012 | Train: 0.7799 | Val: 0.7648 | LR: 0.000100
Epoch 013 | Train: 0.7774 | Val: 0.7657 | LR: 0.000100
Epoch 014 | Train: 0.7768 | Val: 0.7636 | LR: 0.000100
Epoch 015 | Train: 0.7764 | Val: 0.7638 | LR: 0.000100
Epoch 016 | Train: 0.7740 | Val: 0.7628 | LR: 0.000100
Epoch 017 | Train: 0.7740 | Val: 0.7620 | LR: 0.000100
Epoch 018 | Train: 0.7728 | Val: 0.7628 | LR: 0.000100
Epoch 019 

In [ ]:
def generate_graph_sequences_batch(model, target_num_nodes, batch_size=1, max_len=23, eos_token=4, pad_token=5):
    model.eval()
    device = next(model.parameters()).device
    start_token = target_num_nodes + VERTEX_OFFSET
    seqs = torch.full((batch_size, 1), start_token, dtype=torch.long, device=device)
    finished = torch.zeros(batch_size, dtype=torch.bool, device=device)

    with torch.no_grad():
        for _ in range(max_len - 1):
            logits = model(seqs)
            next_token_logits = logits[:, -1, :]

            # Mask invalid tokens
            next_token_logits[:, pad_token:VERTEX_OFFSET] = -float('inf')
            next_token_logits[:, VERTEX_OFFSET:] = -float('inf')

            probs = F.softmax(next_token_logits, dim=-1)
            next_tokens = torch.multinomial(probs, num_samples=1)

            seqs = torch.cat([seqs, next_tokens], dim=1)

            is_eos = (next_tokens.squeeze(-1) == eos_token)
            finished = finished | is_eos

            if finished.all():
                break

    result_seqs = []
    for i in range(batch_size):
        row = seqs[i].tolist()
        if eos_token in row:
            row = row[:row.index(eos_token) + 1]
        result_seqs.append(row)
    return result_seqs


def decode_to_matrix(seq, eos_token=4, pad_token=5):
    clean_seq = [t for t in seq[1:] if t not in (eos_token, pad_token)]
    L = len(clean_seq)
    if L == 0:
        return np.zeros((1, 1), dtype=np.int32)
    N = int(np.floor((1 + np.sqrt(1 + 8 * L)) / 2))
    matrix = np.zeros((N, N), dtype=np.int32)
    idx = 0
    for i in range(1, N):
        for j in range(i):
            if idx < L:
                matrix[i, j] = clean_seq[idx]
                matrix[j, i] = clean_seq[idx]
                idx += 1
    return matrix


# Quick validation check with batching
num_graphs = 100
target_nodes = NUM_VERTICES
valid_graphs = 0

gen_seqs = generate_graph_sequences_batch(model, target_num_nodes=target_nodes, batch_size=num_graphs)
for gen_seq in gen_seqs:
    adj_matrix = decode_to_matrix(gen_seq)
    G = nx.from_numpy_array((adj_matrix > 0).astype(int))
    valid_edges = np.all((adj_matrix >= 0) & (adj_matrix <= 2))
    if G.number_of_nodes() > 0 and nx.is_connected(G) and valid_edges:
        valid_graphs += 1
print(f"Validity check: {valid_graphs}/{num_graphs} generated graphs are valid.")


Validity check: 82/100 generated graphs are valid.


In [ ]:

import subprocess, sys
print("Installing Julia + juliacall...")
subprocess.run([sys.executable, "-m", "pip", "install", "juliacall", "-q"], check=True)

from juliacall import Main as jl
jl.seval("import Pkg")
for pkg in ["ChipFiring", "Graphs", "TreeWidthSolver"]:
    print(f"  Setting up {pkg}...")
    jl.seval(f'if !haskey(Pkg.project().dependencies, "{pkg}") Pkg.add("{pkg}") end')
jl.seval("using ChipFiring; using Graphs; using TreeWidthSolver")
jl.seval('function fast_convert(np_array) return convert(Matrix{Int64}, np_array) end')
fast_convert = jl.fast_convert
print("Julia setup complete!")

N_REWARD = NUM_VERTICES
MYN = N_REWARD * (N_REWARD - 1) // 2
MAX_EDGES_TO_CHECK = 18
TRIU_I, TRIU_J = np.triu_indices(N_REWARD, k=1)
score_cache = {}


def compute_gon_2_subdivision(g, gonality, n, num_nonzero_edges):
    if check_non_gsg(g, gonality, n, num_nonzero_edges):
        return gonality
    sub_g = jl.subdivide(g, 2)
    rank_to_check = gonality
    while rank_to_check > 0:
        if jl.compute_gonality(sub_g, min_d=rank_to_check, max_d=rank_to_check) == -1:
            return rank_to_check + 1
        rank_to_check -= 1
    return jl.compute_gonality(sub_g)


def check_non_gsg(g, gonality, n, num_edges):
    if n <= 5:
        return True
    if gonality <= 4:
        return True
    if num_edges > MAX_EDGES_TO_CHECK:
        return True
    if jl.compute_genus(g) <= 5:
        return True
    return False


def is_connected_fast(edges, n):
    parent = list(range(n))

    def find(i):
        if parent[i] == i:
            return i
        parent[i] = find(parent[i])
        return parent[i]

    def union(i, j):
        ri, rj = find(i), find(j)
        if ri != rj:
            parent[ri] = rj

    for idx in range(len(edges)):
        if edges[idx] > 0:
            union(TRIU_I[idx], TRIU_J[idx])
    return all(find(i) == find(0) for i in range(1, n))


def calcScore(state):
    edges = state[:MYN]
    state_key = tuple(edges)

    if state_key in score_cache:
        return score_cache[state_key]

    num_edges_total_weight = np.sum(edges)
    if num_edges_total_weight == 0 or not is_connected_fast(edges, N_REWARD):
        result = (0.0, False, {"reason": "disconnected_or_empty"})
        score_cache[state_key] = result
        return result

    adj = np.zeros((N_REWARD, N_REWARD), dtype=int)
    adj[TRIU_I, TRIU_J] = edges
    adj[TRIU_J, TRIU_I] = edges

    try:
        jl_matrix = fast_convert(adj)
        graph = jl.ChipFiringGraph(jl_matrix)
        gon = int(jl.compute_gonality(graph))

        if gon < 5:
            result = (
                float(gon),
                False,
                {"gon": gon, "gon2": None, "treewidth": None, "reason": "gon_lt_5"},
            )
            score_cache[state_key] = result
            return result

        gon2 = compute_gon_2_subdivision(graph, gon, N_REWARD, num_edges_total_weight)
        treewidth = int(jl.exact_treewidth(jl.SimpleGraph(graph.adj_matrix)))
        is_gsg = gon != gon2

        reward = (
            50
            + 200 * (gon - gon2)
            + (1000 / num_edges_total_weight)
            - (10 * treewidth)
            + np.random.normal(0, 0.1)
        )

        info = {
            "gon": gon,
            "gon2": gon2,
            "treewidth": treewidth,
            "num_edges": int(num_edges_total_weight),
        }

        result = (reward, is_gsg, info)
        score_cache[state_key] = result
        return result

    except Exception as e:
        result = (0.0, False, {"reason": "error", "error": str(e)})
        score_cache[state_key] = result
        return result


def sequence_to_upper_triangle(seq):
    matrix = decode_to_matrix(seq)
    if matrix.shape[0] != N_REWARD:
        return None, None
    if not np.all((matrix >= 0) & (matrix <= 2)):
        return None, None
    return matrix[TRIU_I, TRIU_J].tolist(), matrix


print("Reward function ready!")

Installing Julia + juliacall...


/usr/local/lib/python3.12/dist-packages/juliacall/__init__.py:61: UserWarning: torch was imported before juliacall. This may cause a segfault. To avoid this, import juliacall before importing torch. For updates, see https://github.com/pytorch/pytorch/issues/78829.
  warnings.warn(


[juliapkg] Found dependencies: /usr/local/lib/python3.12/dist-packages/juliacall/juliapkg.json
[juliapkg] Found dependencies: /usr/local/lib/python3.12/dist-packages/juliapkg/juliapkg.json
[juliapkg] Locating Julia 1.10.3 - 1.11
[juliapkg] WARNING: You have Julia 1.12.6 installed but 1.10.3 - 1.11 is required.
[juliapkg]   It is recommended that you upgrade Julia or install JuliaUp.
[juliapkg] Querying Julia versions from https://julialang-s3.julialang.org/bin/versions.json
[juliapkg] WARNING: About to install Julia 1.11.9 to /root/.julia/environments/pyjuliapkg/pyjuliapkg/install.
[juliapkg]   If you use juliapkg in more than one environment, you are likely to
[juliapkg]   have Julia installed in multiple locations. It is recommended to
[juliapkg]   install JuliaUp (https://github.com/JuliaLang/juliaup) or Julia
[juliapkg]   (https://julialang.org/downloads) yourself.
[juliapkg] Downloading Julia from https://julialang-s3.julialang.org/bin/linux/x64/1.11/julia-1.11.9-linux-x86_64.tar.

   Resolving package versions...
   Installed ArnoldiMethod ─ v0.4.0
   Installed ChipFiring ──── v0.4.1
   Installed Graphs ──────── v1.14.0
    Updating `~/.julia/environments/pyjuliapkg/Project.toml`
  [be096eff] + ChipFiring v0.4.1
    Updating `~/.julia/environments/pyjuliapkg/Manifest.toml`
  [ec485272] + ArnoldiMethod v0.4.0
  [be096eff] + ChipFiring v0.4.1
  [864edb3b] + DataStructures v0.19.4
  [86223c79] + Graphs v1.14.0
  [d25df0c9] + Inflate v0.1.5
  [699a6c99] + SimpleTraits v0.9.6
  [90137ffa] + StaticArrays v1.9.18
  [1e83bf80] + StaticArraysCore v1.4.4
  [10745b16] + Statistics v1.11.1
  [37e2e46d] + LinearAlgebra v1.11.0
  [2f01184e] + SparseArrays v1.11.0
  [e66e0078] + CompilerSupportLibraries_jll v1.1.1+0
  [4536629a] + OpenBLAS_jll v0.3.27+1
  [bea87d4a] + SuiteSparse_jll v7.7.0+0
  [8e850b90] + libblastrampoline_jll v5.11.0+0
Precompiling project...
   1000.5 ms  ✓ StaticArraysCore
   1087.3 ms  ✓ Inflate
   1321.6 ms  ✓ Statistics
   1546.0 ms  ✓ SimpleTraits
   

  Setting up Graphs...
  Setting up TreeWidthSolver...


    Updating `~/.julia/environments/pyjuliapkg/Project.toml`
  [7d267fc5] + TreeWidthSolver v0.3.5
    Updating `~/.julia/environments/pyjuliapkg/Manifest.toml`
  [1520ce14] + AbstractTrees v0.4.5
  [50ba71b6] + BitBasis v0.9.10
  [861a8166] + Combinatorics v1.1.0
  [7d267fc5] + TreeWidthSolver v0.3.5
Precompiling project...
   1286.4 ms  ✓ AbstractTrees
   1955.6 ms  ✓ Combinatorics
   1967.6 ms  ✓ BitBasis
   1230.1 ms  ✓ TreeWidthSolver
  4 dependencies successfully precompiled in 3 seconds. 65 already precompiled.


Julia setup complete!
Reward function ready!


In [ ]:
##############################################################
# 10-TRIAL BENCHMARK
# Run Cells 1-6 first.
##############################################################

import time
import math
from networkx.algorithms.graph_hashing import weisfeiler_lehman_graph_hash


def retrain_model(model, data, epochs=20, lr=1e-4):
    dev = next(model.parameters()).device
    ts = int(0.9 * len(data))
    td, vd = torch.utils.data.random_split(data, [ts, len(data) - ts])
    tl = DataLoader(td, batch_size=32, shuffle=True)
    vl = DataLoader(vd, batch_size=32, shuffle=False)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    crit = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN)
    best = float("inf")
    pat = 0

    for ep in range(epochs):
        model.train()
        for b in tl:
            b = b.to(dev)
            x, y = b[:, :-1], b[:, 1:]
            opt.zero_grad()
            loss = crit(model(x).reshape(-1, 30), y.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        model.eval()
        vloss = 0
        with torch.no_grad():
            for b in vl:
                b = b.to(dev)
                xv, yv = b[:, :-1], b[:, 1:]
                vloss += crit(model(xv).reshape(-1, 30), yv.reshape(-1)).item()

        av = vloss / len(vl)
        if av < best:
            best = av
            pat = 0
        else:
            pat += 1
            if pat >= 3:
                break
    return model


def graph_hash(matrix):
    return weisfeiler_lehman_graph_hash(nx.from_numpy_array(matrix), edge_attr='weight')


NUM_TRIALS = 10
MAX_ROUNDS = 50
GRAPHS_PER_BATCH = 800
TOP_K_PERCENT = 0.05
RETRAIN_EPOCHS = 20
TIME_LIMIT = 600

trial_times = []
trial_timing = []

print("=" * 60)
print(f"  RUNNING {NUM_TRIALS} TRIALS | V={NUM_VERTICES}")
print(f"  Pool approach: score all, retrain on top {TOP_K_PERCENT*100:.0f}%")
print("=" * 60)

for trial in range(1, NUM_TRIALS + 1):
    print(f"\n{'='*60}")
    print(f"  TRIAL {trial}/{NUM_TRIALS}")
    print(f"{'='*60}")

    trial_start = time.time()
    score_cache.clear()
    timing = {
        "data": 0.0,
        "initial_training": 0.0,
        "generation": 0.0,
        "scoring": 0.0,
        "retraining": 0.0,
    }

    # Fresh random training data
    t_data = time.time()
    seqs = []
    while len(seqs) < 3000:
        m = generate_random_graph(NUM_VERTICES, EDGE_PROBS)
        if is_connected(m):
            seqs.append(torch.tensor(matrix_to_sequence(m, NUM_VERTICES), dtype=torch.long))
    ds = torch.stack([F.pad(s, (0, SEQ_LENGTH - len(s)), value=PAD_TOKEN) for s in seqs])
    timing["data"] += time.time() - t_data
    print(f"  Data: {len(ds)} graphs ({timing['data']:.1f}s)")

    # Score all initial random graphs to build the scored pool
    t_score_init = time.time()
    scored_pool = []
    for i, s in enumerate(seqs):
        seq_list = s.tolist()
        ut, mx = sequence_to_upper_triangle(seq_list)
        if ut is not None:
            rw, is_gsg, info = calcScore(ut)
            st = F.pad(s, (0, SEQ_LENGTH - len(s)), value=PAD_TOKEN)
            scored_pool.append((rw, is_gsg, info, st))
    timing["scoring"] += time.time() - t_score_init
    print(f"  Scored initial pool: {len(scored_pool)} graphs ({time.time() - t_score_init:.1f}s)")

    # Fresh model + initial training on top-K% of scored pool
    scored_pool.sort(key=lambda x: x[0], reverse=True)
    top_k_count = max(1, int(len(scored_pool) * TOP_K_PERCENT))
    top_k_seqs = [sp[3] for sp in scored_pool[:top_k_count]]
    train_ds = torch.stack(top_k_seqs)

    t_train = time.time()
    mdl = GraphGPT(vocab_size=30, max_seq_len=SEQ_LENGTH, pad_token=5).to(device)
    opt = optim.AdamW(mdl.parameters(), lr=1e-4, weight_decay=0.01)
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, "min", patience=2, factor=0.5)
    crit = nn.CrossEntropyLoss(ignore_index=5)
    scl = GradScaler()

    ts = int(0.9 * len(train_ds))
    td, vd = torch.utils.data.random_split(train_ds, [ts, len(train_ds) - ts])
    tl = DataLoader(td, batch_size=32, shuffle=True)
    vl = DataLoader(vd, batch_size=32, shuffle=False)

    best_vl = float("inf")
    no_imp = 0
    for ep in range(100):
        mdl.train()
        for b in tl:
            b = b.to(device)
            x, y = b[:, :-1], b[:, 1:]
            opt.zero_grad()
            with autocast():
                loss = crit(mdl(x).reshape(-1, 30), y.reshape(-1))
            scl.scale(loss).backward()
            scl.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
            scl.step(opt)
            scl.update()

        mdl.eval()
        vl_loss = 0
        with torch.no_grad():
            for b in vl:
                b = b.to(device)
                xv, yv = b[:, :-1], b[:, 1:]
                vl_loss += crit(mdl(xv).reshape(-1, 30), yv.reshape(-1)).item()

        av = vl_loss / len(vl)
        sched.step(av)
        if av < best_vl:
            best_vl = av
            no_imp = 0
        else:
            no_imp += 1
            if no_imp >= 5:
                break

    timing["initial_training"] += time.time() - t_train
    print(f"  Initial training done (epoch {ep+1}, val_loss={best_vl:.4f}, {timing['initial_training']:.1f}s)")

    # Generate-score-retrain loop
    found = False
    seen_hashes = set()
    found_round = 0

    for rnd in range(1, MAX_ROUNDS + 1):
        if time.time() - trial_start > TIME_LIMIT:
            print(f"  Time limit at round {rnd}")
            break

        t_gen = time.time()
        gen = []
        dups = 0
        invalid = 0

        # Batched generation
        batch_size = 200
        for i in range(0, GRAPHS_PER_BATCH, batch_size):
            curr_bs = min(batch_size, GRAPHS_PER_BATCH - i)
            batch_sqs = generate_graph_sequences_batch(mdl, target_num_nodes=NUM_VERTICES, batch_size=curr_bs, max_len=SEQ_LENGTH)
            for sq in batch_sqs:
                ut, mx = sequence_to_upper_triangle(sq)
                if ut is None:
                    invalid += 1
                    continue
                h = graph_hash(mx)
                if h in seen_hashes:
                    dups += 1
                seen_hashes.add(h)
                gen.append((sq, mx, ut))
        timing["generation"] += time.time() - t_gen

        t_score = time.time()
        for sq, mx, ut in gen:
            rw, is_gsg, info = calcScore(ut)
            st = torch.tensor(sq, dtype=torch.long)
            if len(st) < SEQ_LENGTH:
                st = F.pad(st, (0, SEQ_LENGTH - len(st)), value=PAD_TOKEN)
            elif len(st) > SEQ_LENGTH:
                st = st[:SEQ_LENGTH]
            scored_pool.append((rw, is_gsg, info, st))

            if is_gsg:
                found = True
                found_round = rnd
                elapsed = time.time() - trial_start
                print(f"\n  GSG FOUND in round {rnd}!")
                print(f"  Time: {elapsed:.1f}s")
                print(f"  Reward: {rw:.2f}")
                print(f"  gon(G): {info['gon']}")
                print(f"  gon(sigma_2(G)): {info['gon2']}")
                print(f"  treewidth: {info['treewidth']}")
                print(f"  Upper triangle: {ut}")
                print(f"  Matrix:\n{mx}")
                break
        timing["scoring"] += time.time() - t_score

        if found:
            break

        # Rank entire pool, retrain on top K%
        scored_pool.sort(key=lambda x: x[0], reverse=True)
        top_k_count = max(1, int(len(scored_pool) * TOP_K_PERCENT))
        top_k_seqs = [sp[3] for sp in scored_pool[:top_k_count]]
        best_reward = scored_pool[0][0]

        print(
            f"  Round {rnd}: generated={len(gen)}, dups={dups}, invalid={invalid}, "
            f"pool={len(scored_pool)}, top_k={top_k_count}, best_rw={best_reward:.2f}"
        )

        t_retrain = time.time()
        mdl = retrain_model(mdl, torch.stack(top_k_seqs), epochs=RETRAIN_EPOCHS)
        timing["retraining"] += time.time() - t_retrain

    t = time.time() - trial_start
    trial_timing.append(timing)

    print(
        "  Timing split: "
        f"data={timing['data']:.1f}s | "
        f"initial_train={timing['initial_training']:.1f}s | "
        f"generation={timing['generation']:.1f}s | "
        f"scoring={timing['scoring']:.1f}s | "
        f"retraining={timing['retraining']:.1f}s"
    )

    if found:
        trial_times.append(t)
        print(f"  Trial {trial}: {t:.1f}s (round {found_round})")
    else:
        trial_times.append(math.inf)
        print(f"  Trial {trial}: FAILED ({t:.1f}s)")


# === FINAL REPORT ===
print("\n" + "=" * 60)
print("  RESULTS SUMMARY")
print("=" * 60)

converged = [t for t in trial_times if t != math.inf]
print(f"\n  All trials:")
for i, t in enumerate(trial_times):
    print(f"    Trial {i+1}: {f'{t:.1f}s' if t != math.inf else 'DID NOT CONVERGE'}")

print(f"\n  Convergence: {len(converged)}/{NUM_TRIALS}")

if converged:
    arr = np.array(converged)
    mean = np.mean(arr)
    std = np.std(arr, ddof=1) if len(arr) > 1 else 0.0
    ci = 1.96 * std / np.sqrt(len(arr)) if len(arr) > 1 else 0.0
    med = np.median(arr)
    p25, p75 = np.percentile(arr, [25, 75])
    print(f"\n  Mean:    {mean:.1f}s +/- {ci:.1f}s (95% CI)")
    print(f"  95% CI:  [{mean - ci:.1f}s, {mean + ci:.1f}s]")
    print(f"  Median:  {med:.1f}s")
    print(f"  IQR:     [{p25:.1f}s, {p75:.1f}s]")
    print(f"  Range:   [{arr.min():.1f}s, {arr.max():.1f}s]")
else:
    print("\n  No successful trials.")

if trial_timing:
    totals = {
        key: sum(t[key] for t in trial_timing)
        for key in ["data", "initial_training", "generation", "scoring", "retraining"]
    }
    measured = sum(totals.values())
    print("\n  Aggregate timing split:")
    for key, value in totals.items():
        pct = (100.0 * value / measured) if measured else 0.0
        print(f"    {key}: {value:.1f}s ({pct:.1f}%)")

print("=" * 60)


  RUNNING 10 TRIALS | V=7
  Pool approach: score all, retrain on top 5%

  TRIAL 1/10
  Data: 3000 graphs (5.4s)
  Scored initial pool: 3000 graphs (1.4s)


/tmp/ipykernel_628/2624666957.py:121: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scl = GradScaler()
/tmp/ipykernel_628/2624666957.py:136: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Initial training done (epoch 32, val_loss=0.9842, 2.7s)
  Round 1: generated=768, dups=3, invalid=32, pool=3768, top_k=188, best_rw=107.03
  Round 2: generated=783, dups=9, invalid=17, pool=4551, top_k=227, best_rw=107.05
  Round 3: generated=784, dups=11, invalid=16, pool=5335, top_k=266, best_rw=107.05
  Round 4: generated=791, dups=7, invalid=9, pool=6126, top_k=306, best_rw=107.05

  GSG FOUND in round 5!
  Time: 98.9s
  Reward: 286.88
  gon(G): 6
  gon(sigma_2(G)): 5
  treewidth: 3
  Upper triangle: [1, 1, 0, 2, 0, 0, 0, 1, 0, 0, 2, 2, 0, 2, 0, 2, 0, 1, 0, 0, 1]
  Matrix:
[[0 1 1 0 2 0 0]
 [1 0 0 1 0 0 2]
 [1 0 0 2 0 2 0]
 [0 1 2 0 2 0 1]
 [2 0 0 2 0 0 0]
 [0 0 2 0 0 0 1]
 [0 2 0 1 0 1 0]]
  Timing split: data=5.4s | initial_train=2.7s | generation=3.0s | scoring=84.6s | retraining=3.2s
  Trial 1: 98.9s (round 5)

  TRIAL 2/10
  Data: 3000 graphs (5.4s)
  Scored initial pool: 3000 graphs (1.6s)
  Initial training done (epoch 53, val_loss=0.9233, 5.0s)
  Round 1: generated=767, d